<a href="https://colab.research.google.com/github/Valdan-D/remorse/blob/main/eda_Danilo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REMORSE — Exploratory Data Analysis
**Reptilian Evaluation of Mesozoic Origins: Retrospective Study on Extinction**

Questo notebook esplora i due dataset fossili della Paleobiology Database (PBDB):
- `dinos.csv` — occorrenze fossili di Dinosauria
- `plants.csv` — occorrenze fossili di Plantae (Mesozoico)

Obiettivo: capire la qualità dei dati, identificare anomalie e prendere decisioni informate sulle trasformazioni da applicare nella pipeline ETL.

## 0. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Librerie caricate.')

## 1. Caricamento dei dataset

In [ ]:
dinos = pd.read_csv('data/raw/dinos.csv', low_memory=False)
plants = pd.read_csv('data/raw/plants.csv', low_memory=False)

print(f'dinos.csv  → {dinos.shape[0]:,} righe, {dinos.shape[1]} colonne')
print(f'plants.csv → {plants.shape[0]:,} righe, {plants.shape[1]} colonne')

## 2. Struttura dei dataset

In [ ]:
print('=== DINOS ===')
dinos.info()

In [ ]:
print('=== PLANTS ===')
plants.info()

In [ ]:
# Colonne presenti in entrambi i dataset
comuni = set(dinos.columns) & set(plants.columns)
solo_dinos = set(dinos.columns) - set(plants.columns)
solo_plants = set(plants.columns) - set(dinos.columns)

print(f'Colonne comuni:      {len(comuni)}')
print(f'Solo in dinos:       {len(solo_dinos)} → {solo_dinos}')
print(f'Solo in plants:      {len(solo_plants)} → {solo_plants}')

## 3. Analisi dei valori nulli

In [ ]:
def null_report(df, name):
    null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]
    print(f'\n=== {name} — colonne con valori nulli ===')
    print(null_pct.to_string())

null_report(dinos, 'DINOS')
null_report(plants, 'PLANTS')

In [ ]:
# Percentuale di record con coordinate valide
dinos_coord = dinos[['lat', 'lng']].dropna()
plants_coord = plants[['lat', 'lng']].dropna()

print(f'Dinos con coordinate valide:  {len(dinos_coord):,} ({len(dinos_coord)/len(dinos)*100:.1f}%)')
print(f'Plants con coordinate valide: {len(plants_coord):,} ({len(plants_coord)/len(plants)*100:.1f}%)')

## 4. Duplicati

In [ ]:
print(f'Duplicati in dinos:  {dinos.duplicated().sum()}')
print(f'Duplicati in plants: {plants.duplicated().sum()}')

# Verifica unicità di occurrence_no
print(f'\noccurrence_no univoci in dinos:  {dinos["occurrence_no"].nunique():,} / {len(dinos):,}')
print(f'occurrence_no univoci in plants: {plants["occurrence_no"].nunique():,} / {len(plants):,}')

## 5. Distribuzione per periodo geologico

In [ ]:
print('=== DINOS — early_interval (top 20) ===')
print(dinos['early_interval'].value_counts().head(20))

In [ ]:
print('=== PLANTS — early_interval (top 20) ===')
print(plants['early_interval'].value_counts().head(20))

In [ ]:
# Distribuzione per era (max_ma come proxy)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

dinos['max_ma'].dropna().hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Dinos — distribuzione età (max_ma)')
axes[0].set_xlabel('Milioni di anni fa')
axes[0].set_ylabel('Occorrenze')
axes[0].invert_xaxis()

plants['max_ma'].dropna().hist(bins=50, ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Plants — distribuzione età (max_ma)')
axes[1].set_xlabel('Milioni di anni fa')
axes[1].invert_xaxis()

plt.tight_layout()
plt.show()

## 6. Distribuzione geografica

In [ ]:
# Top 15 paesi per numero di ritrovamenti
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

dinos['cc'].value_counts().head(15).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Dinos — top 15 paesi')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=45)

plants['cc'].value_counts().head(15).plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('Plants — top 15 paesi')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Classificazione tassonomica

In [ ]:
print('=== DINOS — classi più frequenti ===')
print(dinos['class'].value_counts().head(10))

print('\n=== DINOS — Aves nel dataset ===')
aves = dinos[dinos['class'] == 'Aves']
print(f'Record Aves: {len(aves):,} ({len(aves)/len(dinos)*100:.1f}%)')

In [ ]:
print('=== PLANTS — phylum più frequenti ===')
print(plants['phylum'].value_counts().head(10))

## 8. Statistiche descrittive su max_ma / min_ma

In [ ]:
for name, df in [('DINOS', dinos), ('PLANTS', plants)]:
    print(f'\n=== {name} ===')
    print(df[['max_ma', 'min_ma']].describe())

## 9. Decisioni di trasformazione

Compilare questa sezione al termine dell'EDA con le decisioni prese dal gruppo.

| Trasformazione | Decisione | Note |
|---|---|---|
| Escludere Aves | Da decidere | Vedere % in sezione 7 |
| Gestione nulli | Da decidere | Vedere sezione 3 |
| Eliminazione duplicati | Sì | Vedere sezione 4 |
| Colonna `mid_ma` | Sì | `(max_ma + min_ma) / 2` |
| Colonna `period_group` | Sì | Triassic / Jurassic / Cretaceous |
| Colonna `dataset_type` | Sì | Dinosauria / Plantae |
| Flag `has_valid_coords` | Sì | Booleano su lat/lng |
| Normalizzazione paesi | Da decidere | Vedere distribuzione in sezione 6 |

## 10. Salvataggio e commit su GitHub

In [ ]:
# Eseguire al termine della sessione per salvare il notebook nel repo
!git add notebooks/eda.ipynb
!git commit -m "notebooks: aggiornato EDA"
!git push